# Step 11 — TPE Hyperparameter Optimisation

Drop-in continuation of **ae_estimation_ann.ipynb**.  
Requires these objects already in the kernel (built by Steps 1–6/8/10):

| Object | Origin |
|---|---|
| `dataset_seg`, `cut_param`, `df_info`, `data_file_path` | Steps 2/3 |
| `X_train_sc`, `X_test_sc`, `y_train_sc`, `y_test_sc` | Step 6 |
| `scaler_X`, `scaler_y` | Step 6 |
| `AeEstimatorMLP`, `MillingDataset` | Steps 7/8 |
| `mc_predict` | Step 10 |
| `N_SEGMENTS`, `KC11_NOMINAL`, `MC_NOMINAL`, `FS` | Steps 2/6 |
| `build_feature_vector` | Step 5 |

Three hyperparameters are jointly optimised with Bayesian TPE:

| Parameter | Search space | Physical rationale |
|---|---|---|
| `n_layers` | 1–4 | Depth of the ae–torque nonlinear mapping |
| `units_lN` | 32–192 (step 32) | Width bounded for your small-N dataset |
| `dropout` | 0.05–0.50 | Controls MC Dropout quality; non-monotone |
| `n_segments` | 2–8 | Window count per cut (optional axis) |


## 11.0 · Install Optuna (run once)

In [ ]:
# Uncomment first time only
!pip install optuna --quiet


## 11.1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner

optuna.logging.set_verbosity(optuna.logging.WARNING)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 11.2 · Segment-count helper

`n_segments` is searched as the third HPO axis. This function re-runs the
exact segmentation logic from your Step 2 for any requested count and returns
scaled train/test splits. It calls your existing `build_feature_vector`,
`scaler_X`, and `scaler_y` — no physics logic is duplicated here.

Set `SEARCH_N_SEGMENTS = False` in Section 11.4 to skip this and reuse
the already-built `X_train_sc`/`y_train_sc` (faster first run).


In [ ]:
def build_scaled_splits_for_n_seg(n_segments: int, random_state: int = 42):
    """
    Re-segment all cuts into `n_segments` equal windows.
    Mirrors the loop in Step 2 exactly, using globals:
      cut_param, df_info, data_file_path,
      build_feature_vector, scaler_X, scaler_y,
      KC11_NOMINAL, MC_NOMINAL, FS
    Returns: X_tr_sc, X_te_sc, y_tr_sc, y_te_sc
    """
    ds_local = []

    for idx in range(len(cut_param)):
        csv_path = data_file_path + "\\all\\" + df_info["SinuTraceFile (*.csv)"][idx]
        df_raw   = pd.read_csv(csv_path)
        cpp      = cut_param.iloc[idx]

        f      = cpp["f (mm/min)"]
        r_tool = cpp["R (mm)"]

        # ramp-up offset (same as Step 2)
        t_ramp_up        = (2 * r_tool + 5) / f * 60
        idx_start        = np.argmax(df_raw["time"] > t_ramp_up)

        # friction baseline (same as Step 2)
        t_friction       = (r_tool + 5) / f * 60
        idx_frict        = np.argmax(df_raw["time"] > t_friction)
        col              = "+/Nck/!SD/nckServoDataActCurr32 [u1; 4]"
        mean_iq_frict    = np.mean(df_raw[col][:idx_frict])

        iq_eval = np.asarray(df_raw[col][idx_start:] - mean_iq_frict, dtype=float)

        seg_len = len(iq_eval) // n_segments
        if seg_len == 0:
            continue  # signal too short for this n_segments — skip

        segs = iq_eval[:seg_len * n_segments].reshape(n_segments, seg_len)
        for seg in segs:
            ds_local.append({
                "signal":        seg,
                "ap":            float(cpp["Ap (mm)"]),
                "fz":            float(cpp["fz"]),
                "n":             float(cpp["N (rpm)"]),
                "D":             float(2 * cpp["R (mm)"]),
                "z":             float(cpp["Z"]),
                "ae":            float(cpp["Ae (mm)"]),
                "mean_iq_frict": float(mean_iq_frict),
            })

    X_loc = np.array([
        build_feature_vector(s, KC11_NOMINAL, MC_NOMINAL, FS) for s in ds_local
    ])
    y_loc = np.array([s["ae"] for s in ds_local])

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_loc, y_loc, test_size=0.2, random_state=random_state
    )
    X_tr_sc = scaler_X.transform(X_tr)
    X_te_sc = scaler_X.transform(X_te)
    y_tr_sc = scaler_y.transform(y_tr.reshape(-1, 1)).ravel()
    y_te_sc = scaler_y.transform(y_te.reshape(-1, 1)).ravel()
    return X_tr_sc, X_te_sc, y_tr_sc, y_te_sc

print("build_scaled_splits_for_n_seg() ready.")


## 11.3 · Training helper

Mirrors your Step 8 loop (Adam + HuberLoss delta=1.0, batch=16).
The only addition is `trial.report()` and `trial.should_prune()`
for Hyperband early stopping.


In [ ]:
def train_trial(
    model, X_tr, y_tr, X_te, y_te,
    n_epochs=300, batch_size=16, lr=1e-3,
    trial=None
):
    """
    Train and return best validation Huber loss.
    Reports val loss per epoch to Optuna for HyperbandPruner.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.HuberLoss(delta=1.0)   # same as Step 8

    tr_loader  = DataLoader(MillingDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(MillingDataset(X_te, y_te), batch_size=len(X_te))

    best_val = float("inf")
    for epoch in range(n_epochs):
        model.train()
        for X_b, y_b in tr_loader:
            optimizer.zero_grad()
            criterion(model(X_b), y_b).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for X_v, y_v in val_loader:
                val_loss = criterion(model(X_v), y_v).item()

        best_val = min(best_val, val_loss)

        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    return best_val

print("train_trial() ready.")


## 11.4 · Optuna objective function

Set `SEARCH_N_SEGMENTS = True` to include segment count in the search.  
Set `SEARCH_N_SEGMENTS = False` to reuse the existing `X_train_sc`/`y_train_sc`
(recommended for a first architecture-only sweep — much faster).


In [ ]:
# ── configuration ─────────────────────────────────────────────────────────
SEARCH_N_SEGMENTS = True    # set False to skip segment rescan
N_EPOCHS_TRIAL    = 150     # budget per trial; full 300 only for final retrain
INPUT_DIM         = X_train_sc.shape[1]   # 13 features from Step 6
# ──────────────────────────────────────────────────────────────────────────

def objective(trial):
    # architecture
    n_layers = trial.suggest_int("n_layers", 1, 4)
    hidden   = [trial.suggest_int(f"units_l{i}", 32, 192, step=32)
                for i in range(n_layers)]
    dropout  = trial.suggest_float("dropout", 0.05, 0.50)

    # segment count (optional)
    if SEARCH_N_SEGMENTS:
        n_seg = trial.suggest_int("n_segments", 2, 8)
        X_tr, X_te, y_tr, y_te = build_scaled_splits_for_n_seg(n_seg)
    else:
        X_tr, X_te = X_train_sc, X_test_sc
        y_tr, y_te = y_train_sc, y_test_sc

    model = AeEstimatorMLP(
        n_features=INPUT_DIM, hidden=hidden, dropout=dropout
    ).to(DEVICE)

    return train_trial(
        model, X_tr, y_tr, X_te, y_te,
        n_epochs=N_EPOCHS_TRIAL, trial=trial
    )

print("objective() ready.")


## 11.5 · Run the study

TPE uses `n_startup_trials=20` random samples before its surrogate kicks in.
`HyperbandPruner` kills clearly bad trials well before epoch 150,
so 80 trials cost roughly the same wall-clock as 20–25 full runs.

> **Quick sanity check first**: set `n_trials=15` and `N_EPOCHS_TRIAL=40`
> above to confirm the loop runs end-to-end before committing to 80 trials.


In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="ae_mlp_tpe",
    sampler=TPESampler(
        n_startup_trials=20,
        multivariate=True,   # models joint layer-dropout-n_seg coupling
        seed=42,
    ),
    pruner=HyperbandPruner(
        min_resource=10,
        max_resource=N_EPOCHS_TRIAL,
        reduction_factor=3,
    ),
)

study.optimize(
    objective,
    n_trials=80,
    show_progress_bar=True,
    gc_after_trial=True,
)

bt = study.best_trial
print(f"Best val loss : {bt.value:.6f}")
print(f"Best params   : {bt.params}")


## 11.6 · Results summary

In [ ]:
# top-10 table
trials_df = study.trials_dataframe()
completed = trials_df[trials_df["state"] == "COMPLETE"].sort_values("value")
pcols = [c for c in completed.columns if c.startswith("params_")]
print("Top 10 trials:")
print(completed[["number", "value"] + pcols].head(10).to_string(index=False))

# optimisation history + dropout scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vals = [t.value for t in study.trials if t.value is not None]
best_so_far = [min(vals[:i+1]) for i in range(len(vals))]
axes[0].scatter(range(len(vals)), vals, s=12, alpha=0.4, color="steelblue", label="Trial")
axes[0].plot(best_so_far, color="coral", lw=1.5, label="Best so far")
axes[0].axvline(20, color="gray", ls="--", lw=0.8, label="TPE start (trial 20)")
axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Val loss (Huber)")
axes[0].set_title("Optimisation history")
axes[0].legend(fontsize=9); axes[0].grid(True, lw=0.4)

sc = axes[1].scatter(
    completed["params_dropout"], completed["value"],
    c=completed["params_n_layers"], cmap="viridis", s=20, alpha=0.7
)
plt.colorbar(sc, ax=axes[1], label="n_layers")
axes[1].set_xlabel("Dropout rate"); axes[1].set_ylabel("Val loss (Huber)")
axes[1].set_title("Dropout vs loss  (colour = depth)")
axes[1].grid(True, lw=0.4)

plt.tight_layout()
plt.savefig("tpe_history.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# fANOVA importance — needs at least 30 completed trials
try:
    importances = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame({
        "param":      list(importances.keys()),
        "importance": list(importances.values())
    }).sort_values("importance")

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(imp_df["param"], imp_df["importance"], color="steelblue", height=0.5)
    ax.set_xlabel("fANOVA importance")
    ax.set_title("Hyperparameter importance")
    ax.grid(True, axis="x", lw=0.4)
    plt.tight_layout(); plt.show()
    print(imp_df.sort_values("importance", ascending=False).to_string(index=False))
except Exception as e:
    print(f"Importance unavailable (need 30+ completed trials): {e}")


## 11.7 · Retrain final model

Uses best hyperparameters, full train+test data combined,
and the 300-epoch budget from Step 8.
MC Dropout evaluation is identical to Step 10.


In [ ]:
best_p       = study.best_params
n_lay_opt    = best_p["n_layers"]
hidden_opt   = [best_p[f"units_l{i}"] for i in range(n_lay_opt)]
dropout_opt  = best_p["dropout"]
n_seg_opt    = best_p.get("n_segments", N_SEGMENTS)  # falls back if not searched

print(f"Architecture : {hidden_opt}")
print(f"Dropout      : {dropout_opt:.3f}")
print(f"n_segments   : {n_seg_opt}")

# data at optimal segment count
if SEARCH_N_SEGMENTS:
    X_tr_f, X_te_f, y_tr_f, y_te_f = build_scaled_splits_for_n_seg(n_seg_opt)
else:
    X_tr_f, X_te_f = X_train_sc, X_test_sc
    y_tr_f, y_te_f = y_train_sc, y_test_sc

# combine all data for final training
X_full = np.concatenate([X_tr_f, X_te_f])
y_full = np.concatenate([y_tr_f, y_te_f])

model_opt = AeEstimatorMLP(
    n_features=INPUT_DIM, hidden=hidden_opt, dropout=dropout_opt
).to(DEVICE)

# full 300-epoch run — no trial object, so no pruning
train_trial(model_opt, X_full, y_full, X_te_f, y_te_f, n_epochs=300)
print("Final model trained.")


In [ ]:
# MC Dropout evaluation (mc_predict from Step 10)
x_te_t  = torch.tensor(X_te_f, dtype=torch.float32).to(DEVICE)
ae_mean, ae_std = mc_predict(model_opt, x_te_t)

y_te_phys = scaler_y.inverse_transform(y_te_f.reshape(-1, 1)).ravel()

mae  = mean_absolute_error(y_te_phys, ae_mean)
rmse = float(np.sqrt(np.mean((y_te_phys - ae_mean)**2)))
r2   = r2_score(y_te_phys, ae_mean)
print(f"Optimised model  |  MAE={mae:.4f} mm  |  RMSE={rmse:.4f} mm  |  R2={r2:.4f}")

# parity plot with MC Dropout uncertainty bars
fig, ax = plt.subplots(figsize=(5, 5))
lims = [
    min(y_te_phys.min(), ae_mean.min()) - 0.5,
    max(y_te_phys.max(), ae_mean.max()) + 0.5
]
ax.plot(lims, lims, "k--", lw=1)
ax.errorbar(
    y_te_phys, ae_mean, yerr=1.96 * ae_std,
    fmt="o", ms=5, alpha=0.7, color="steelblue",
    ecolor="lightblue", elinewidth=0.8, capsize=2,
    label="Predicted +/- 95% MC-CI"
)
ax.set_xlabel("Measured ae [mm]")
ax.set_ylabel("Predicted ae [mm]")
ax.set_title(f"Optimised MLP  |  MAE={mae:.3f} mm   R2={r2:.3f}")
ax.legend(fontsize=9); ax.grid(True, lw=0.4)
plt.tight_layout()
plt.savefig("tpe_parity_optimised.png", dpi=150, bbox_inches="tight")
plt.show()


## 11.8 · Persist study and model

In [ ]:
import joblib

joblib.dump(study, "ae_mlp_tpe_study.pkl")

torch.save({
    "model_state_dict": model_opt.state_dict(),
    "hidden_dims":      hidden_opt,
    "dropout":          dropout_opt,
    "n_segments":       n_seg_opt,
    "input_dim":        INPUT_DIM,
    "best_val_loss":    study.best_value,
    "best_params":      best_p,
}, "ae_mlp_best.pt")

print("Saved  ae_mlp_tpe_study.pkl")
print("Saved  ae_mlp_best.pt")
